# Lesson 1: Local Shared Memory

## Objective

Start Geond with local PostgreSQL, seed sample evidence, search it, and smoke-test the MCP surface.

## Prerequisites

- Python 3.11+
- `uv`
- Docker with Compose
- Run this notebook from the repository root.

## Safety

This lesson uses local Docker PostgreSQL and sample data. It does not require private transcripts, API keys, or cloud credentials.


In [ ]:
import subprocess
from pathlib import Path

REPO = Path.cwd()


def run(args, check=True):
    print("$", " ".join(args))
    result = subprocess.run(args, cwd=REPO, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed: {result.returncode}")
    return result

## Run: start local storage

Expected outcome: the `geond-postgres` container is running and ready for migrations.


In [ ]:
run(["docker", "compose", "up", "-d", "postgres"])
run(["docker", "compose", "--profile", "tools", "run", "--rm", "geond-migrate"])

## Run: verify and seed sample memory

Expected outcome: `doctor` reports OK and `seed-sample` returns a workspace id for `file:///sample/geond`.


In [ ]:
run(["uv", "run", "geond", "doctor", "--format", "text"])
run(["uv", "run", "geond", "seed-sample"])

## Run: search and smoke-test MCP

Expected outcome: keyword search returns compact evidence, and `mcp-smoke` confirms stdio MCP initialization, resources, and search.


In [ ]:
run(
    [
        "uv",
        "run",
        "geond",
        "search",
        "app_context",
        "--workspace-uri",
        "file:///sample/geond",
        "--mode",
        "keyword",
    ]
)
run(["uv", "run", "geond", "mcp-smoke", "--format", "text", "--strict"])

## Cleanup

When you want a clean sample state, purge the tutorial workspace:

```bash
uv run geond purge-workspace file:///sample/geond --yes
```
